# Experiment: Per-tenant retrieval and chunking ablation

**Question:** does one chunking strategy perform equally well for a payments FAQ and a device manual?

**Success criteria:** compare top-1 expected-fact coverage per tenant, preserve a 100% isolation pass rate, and avoid hiding tenant asymmetry inside one global average. This notebook uses the deterministic lexical retriever from Project 3 so it runs without an API key; it is an offline proxy, not an embedding-model benchmark.


In [1]:
# Setup: project-local imports and reproducibility
from __future__ import annotations

import hashlib
import json
import random
import sys
from collections import defaultdict
from pathlib import Path

SEED = 7
random.seed(SEED)
ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "3-Multi-Tenant-Evals").exists()
)
PROJECT = ROOT / "3-Multi-Tenant-Evals"
sys.path.insert(0, str(PROJECT))

from eval_harness.cli import load_cases
from eval_harness.corpus import load_corpus
from eval_harness.models import Chunk
from eval_harness.retrieval import retrieve, rewrite_query

cases = load_cases(PROJECT / "evals")
section_collections = load_corpus(PROJECT / "knowledge")
{"seed": SEED, "cases": len(cases), "tenants": sorted(section_collections)}


{'seed': 7, 'cases': 8, 'tenants': ['tenant-A', 'tenant-B']}

## Plan

- Hold corpus, questions, contextual rewrite, top-k, and lexical ranking constant.
- Compare heading-aware chunks with three fixed-window size/overlap pairs.
- Measure expected-fact coverage in the first retrieved chunk for each tenant.
- Re-check the tenant field of every hit; a quality gain cannot trade away isolation.


In [2]:
def fixed_windows(text: str, *, size: int, overlap: int) -> list[str]:
    """Create deterministic character windows with an explicit overlap."""
    if overlap >= size:
        raise ValueError("overlap must be smaller than size")
    step = size - overlap
    return [text[start : start + size] for start in range(0, len(text), step)]


def fixed_collections(size: int, overlap: int) -> dict[str, list[Chunk]]:
    collections = {}
    for path in sorted((PROJECT / "knowledge").glob("*.md")):
        tenant_id = path.stem
        chunks = []
        for index, content in enumerate(
            fixed_windows(path.read_text(encoding="utf-8"), size=size, overlap=overlap)
        ):
            digest = hashlib.sha256(content.encode()).hexdigest()[:12]
            chunks.append(
                Chunk(
                    chunk_id=f"{tenant_id}:fixed:{index}:{digest}",
                    tenant_id=tenant_id,
                    source=path.name,
                    section="fixed-window",
                    content=content,
                )
            )
        collections[tenant_id] = chunks
    return collections


strategies = {
    "heading-aware": section_collections,
    "fixed-180-0": fixed_collections(180, 0),
    "fixed-260-40": fixed_collections(260, 40),
    "fixed-420-80": fixed_collections(420, 80),
}
{name: {tenant: len(chunks) for tenant, chunks in corpus.items()} for name, corpus in strategies.items()}


{'heading-aware': {'tenant-A': 4, 'tenant-B': 4},
 'fixed-180-0': {'tenant-A': 6, 'tenant-B': 6},
 'fixed-260-40': {'tenant-A': 5, 'tenant-B': 5},
 'fixed-420-80': {'tenant-A': 3, 'tenant-B': 3}}

In [3]:
def score_strategy(name: str, collections: dict[str, list[Chunk]]) -> list[dict[str, object]]:
    rows = []
    for case in cases:
        query = rewrite_query(case.question, case.history, enabled=True)
        hit = retrieve(collections[case.tenant_id], query, top_k=1)[0]
        coverage = sum(
            term.casefold() in hit.content.casefold() for term in case.expected_terms
        ) / len(case.expected_terms)
        rows.append(
            {
                "strategy": name,
                "tenant": case.tenant_id,
                "case": case.case_id,
                "coverage": coverage,
                "isolated": hit.tenant_id == case.tenant_id,
            }
        )
    return rows


raw_rows = [row for name, corpus in strategies.items() for row in score_strategy(name, corpus)]
grouped = defaultdict(list)
for row in raw_rows:
    grouped[(row["strategy"], row["tenant"])].append(row)

summary = []
for (strategy, tenant), rows in sorted(grouped.items()):
    summary.append(
        {
            "strategy": strategy,
            "tenant": tenant,
            "top1_fact_coverage": round(sum(row["coverage"] for row in rows) / len(rows), 3),
            "isolation_pass_rate": sum(row["isolated"] for row in rows) / len(rows),
        }
    )
summary


[{'strategy': 'fixed-180-0',
  'tenant': 'tenant-A',
  'top1_fact_coverage': 0.5,
  'isolation_pass_rate': 1.0},
 {'strategy': 'fixed-180-0',
  'tenant': 'tenant-B',
  'top1_fact_coverage': 0.5,
  'isolation_pass_rate': 1.0},
 {'strategy': 'fixed-260-40',
  'tenant': 'tenant-A',
  'top1_fact_coverage': 0.375,
  'isolation_pass_rate': 1.0},
 {'strategy': 'fixed-260-40',
  'tenant': 'tenant-B',
  'top1_fact_coverage': 0.625,
  'isolation_pass_rate': 1.0},
 {'strategy': 'fixed-420-80',
  'tenant': 'tenant-A',
  'top1_fact_coverage': 1.0,
  'isolation_pass_rate': 1.0},
 {'strategy': 'fixed-420-80',
  'tenant': 'tenant-B',
  'top1_fact_coverage': 1.0,
  'isolation_pass_rate': 1.0},
 {'strategy': 'heading-aware',
  'tenant': 'tenant-A',
  'top1_fact_coverage': 0.75,
  'isolation_pass_rate': 1.0},
 {'strategy': 'heading-aware',
  'tenant': 'tenant-B',
  'top1_fact_coverage': 1.0,
  'isolation_pass_rate': 1.0}]

## Results and decision

Read the table above per tenant, not only per strategy. A fixed window can split a fact from its condition, while a heading-aware chunk can become too broad when a manual section contains several procedures. The useful production decision is therefore to tune by document shape and retain per-tenant gates.

The experiment is intentionally small. It establishes a reproducible ablation pattern and a quality contract; it does not establish that lexical overlap is a substitute for dense retrieval.


In [4]:
# Record a compact artifact that can be copied into a design review.
assert all(row["isolation_pass_rate"] == 1.0 for row in summary)
best_by_tenant = {}
for tenant in sorted({row["tenant"] for row in summary}):
    candidates = [row for row in summary if row["tenant"] == tenant]
    best_by_tenant[tenant] = max(candidates, key=lambda row: row["top1_fact_coverage"])
result = {
    "metric": "top1_expected_fact_coverage",
    "best_by_tenant": best_by_tenant,
    "all_isolation_checks_passed": True,
}
print(json.dumps(result, indent=2))


{
  "metric": "top1_expected_fact_coverage",
  "best_by_tenant": {
    "tenant-A": {
      "strategy": "fixed-420-80",
      "tenant": "tenant-A",
      "top1_fact_coverage": 1.0,
      "isolation_pass_rate": 1.0
    },
    "tenant-B": {
      "strategy": "fixed-420-80",
      "tenant": "tenant-B",
      "top1_fact_coverage": 1.0,
      "isolation_pass_rate": 1.0
    }
  },
  "all_isolation_checks_passed": true
}


## Next steps

- Repeat the same matrix with `text-embedding-3-small` and one alternative embedding model.
- Add latency and index-size columns before choosing a production default.
- Expand both tenant golden sets before treating a small score delta as meaningful.
- Promote per-tenant thresholds into CI only after the live embedding run is stable.
